In [ ]:
import re
import ast
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\ADVAN WORKPRO\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [ ]:
df = pd.read_csv("data_raw.csv")

print("Shape:", df.shape)
print(df.head(5))

Shape: (14945, 10)
                      Title  \
0          Ayam Woku Manado   
1  Ayam goreng tulang lunak   
2          Ayam cabai kawin   
3               Ayam Geprek   
4               Minyak Ayam   

                                         Ingredients  \
0  1 Ekor Ayam Kampung (potong 12)--2 Buah Jeruk ...   
1  1 kg ayam (dipotong sesuai selera jangan kecil...   
2  1/4 kg ayam--3 buah cabai hijau besar--7 buah ...   
3  250 gr daging ayam (saya pakai fillet)--Secuku...   
4  400 gr kulit ayam & lemaknya--8 siung bawang p...   

                                               Steps  Loves  \
0  1) Cuci bersih ayam dan tiriskan. Lalu peras j...      1   
1  1) Haluskan bumbu2nya (BaPut, ketumbar, kemiri...      1   
2  1) Panaskan minyak di dalam wajan. Setelah min...      2   
3  1) Goreng ayam seperti ayam krispi\n2) Ulek se...     10   
4  1) Cuci bersih kulit ayam. Sisihkan\n2) Ambil ...      4   

                                                 URL Category  \
0  https://co

In [ ]:
# Lower case semua kolom teks
df["Title"] = df["Title"].str.lower()
df["Title Cleaned"] = df["Title Cleaned"].str.lower()
df["Ingredients Cleaned"] = df["Ingredients Cleaned"].str.lower()

In [ ]:
# Stopwords & normalisasi bahan
STOPWORDS = {
    "ikat", "secukupnya", "sesuai", "selera",
    "cincang", "halus", "kasar", "geprek",
    "memarkan", "iris", "tipis", "potong",
    "dadu", "serong", "lembar", "buah",
    "butir", "siung", "sendok", "sdt", "sdm",
    "gram", "kg", "ml", "liter", "cup",
    "gelas", "bungkus", "batang", "ruas",
    "ambil", "kupas", "cuci", "rebus",
    "goreng", "sangrai", "parut", "untuk",
    "tambahan", "taburan",
    "bks", "cm", "uk", "dan", "atau",
    "dengan", "bagian", "cairkan",
    "bubuk", "toping", "topping",
    "lalap", "kulit", "isi",
    "besar", "kecil", "sedang",
    "btg", "btr", "aku", "beli",
    "sedikit", "sck", "ekor", "air"
}

REMOVE_PHRASES = {
    "aku beli", "bahan sambal", "penyedap rasa",
    "garam dan", "maizena cairkan", "tomat uk",
    "garam sck", "teh b", "mangkok makaroni",
    "sedikit jahe", "sedikit gula", "sedikit garam"
}

NORMALIZATION = {
    "cabe": "cabai",
    "cabe ijo": "cabai hijau",
    "ijo": "hijau",
    "saos": "saus",
    "salam": "daun salam",
    "bputih": "bawang putih",
    "bmerah": "bawang merah",
    "bamer": "bawang merah",
    "baput": "bawang putih",
    "btg sereh": "sereh",
    "btr jeruk": "jeruk",
    "grm": "garam",
    "daun": "",        # hanya jika berdiri sendiri
}

REMOVE_WORDS = {"dipotong", "potong", "haluskan", "ruas", "jari", "batang"}


def clean_ingredients(text):
    """Tahap 1 – raw text → list kata bersih."""
    if pd.isna(text):
        return []

    text = str(text).lower()

    # hapus angka & satuan umum
    text = re.sub(r'\d+|\b(gr|kg|sdm|sdt|buah|butir|lembar|siung)\b', '', text)

    # ganti separator '--' menjadi ','
    text = text.replace('--', ',')

    # hapus karakter selain huruf & koma
    text = re.sub(r'[^a-zA-Z, ]', '', text)

    items = text.split(',')

    result = []
    for item in items:
        words = item.split()
        words = [w for w in words if w not in REMOVE_WORDS]
        item_clean = ' '.join(words[:2])   # ambil max 2 kata
        item_clean = item_clean.strip()
        if item_clean and item_clean not in STOPWORDS:
            result.append(item_clean)

    return result


df['Ingredients Cleaned'] = df['Ingredients'].apply(clean_ingredients)
print(df['Ingredients Cleaned'].head(3))

0    [ekor ayam, jeruk nipis, garam, kunyit, bawang...
1    [ayam sesuai, serai memarkan, daun jeruk, bawa...
2    [ayam, cabai hijau, cabai merah, bawang putih,...
Name: Ingredients Cleaned, dtype: object


In [ ]:
# Kata & pola yang ingin dihapus dari judul
UNWANTED_WORDS = {
    "menu", "super", "enak", "mantap", "praktis", "simple", "sederhana",
    "mudah", "hemat", "ekonomi", "murah", "special", "spesial", "instan",
    "kilat", "cepat", "express", "yummy", "maknyus", "endess", "endol",
    "joss", "wuenak", "uenakk", "uenak", "nyuss", "legit", "banget",
    "debm", "ulalaaa"
}

MESSY_PATTERNS = [
    # Subjektif / promosi
    r'\benak+\b', r'\byumm+y?\b', r'\bmaknyus+\b', r'\bendess*\b',
    r'\bendol+\b', r'\bjoss\b', r'\bwuenak+\b', r'\buenakk*\b',
    r'\buenak+\b', r'\bmantap+\b', r'\bnyuss*\b', r'\blegit\b',
    r'\bgurih\b', r'\bgampang\b', r'\bpraktis\b', r'\bsimpel\b',
    r'\bsimple\b', r'\bsederhana\b', r'\bmudah\b', r'\bhemat\b',
    r'\bekonomi[s]?\b', r'\bmurah\b', r'\bspecial\b', r'\bspesial\b',
    r'\binstan[t]?\b', r'\bkilat\b', r'\bcepat\b', r'\bexpress\b',
    r'\bekspres+\b', r'\brenyah\b', r'\bgaring\b', r'\bkrenyes+\b',
    r'\bkress+\b', r'\bhome\s*made\b', r'\brumahan\b', r'\bsehat\b',
    r'\bdiet\b', r'\benaaak\b', r'\bbangeet\b', r'\bramah\b',
    r'\banak\b', r'\bcocok\b', r'\btidak\b', r'\bga\b', r'\bgak\b',
    r'\btanpa\b', r'\bpedass\b', r'\btkl\b', r'\bmasak\b',
    # Nama / "ala" / "by"
    r'\bala\s+[\w\s]{1,20}', r'\bby\s+[\w\s]{1,20}', r'\bala\b', r'\bkw\b',
    # Alat masak & keterangan
    r'\(.*?\)', r'\bteflon\b', r'\bhappy\s*call\b', r'\boven\b',
    r'\bmagic\s*com\b', r'\b(ma?gi?|me?ji?)com\b', r'\bpresto\b',
    r'\brice?\s*cooker\b', r'\bblender\b', r'\bmicrowave\b',
    r'\bmejikom\b', r'\bpenggorengan\b',
    # Kata tambahan
    r'\bno\s+(msg|ribet|santan)\b', r'\bnon\s+msg\b',
    r'\banti\s+gagal\b', r'\bversi\s+\w+', r'\bresep\s+\w+',
    r'\btips+\b', r'\bstep\s+by\s+step\b', r'\bpart\s+\d+',
    r'\bsuka[\s-]*suka\b', r'\bseadanya\b', r'\balakadarnya\b',
    r'\brecook\b', r'\bpr_\w+', r'\ba\s*k\s*a\b', r'\bvs\b',
    # Simbol
    r'[`\'\"\\\/\(\)\[\]{}]', r'[\+\*\!]{2,}', r'[^\w\s]'
]


def clean_title(text):
    """Bersihkan judul: lowercase → hapus pola messy → hapus unwanted words → rapikan spasi."""
    text = str(text).lower()

    # Hapus hashtag
    text = re.sub(r'#\w+', ' ', text)

    # Terapkan semua messy patterns secara berurutan
    for pattern in MESSY_PATTERNS:
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)

    # Hapus unwanted words (whole-word match)
    for word in UNWANTED_WORDS:
        text = re.sub(r'\b' + re.escape(word) + r'\b', ' ', text, flags=re.IGNORECASE)

    # Rapikan spasi
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# Hapus baris yang mengandung missing value walaupun hanya pada salah satu kolom
print("Jumlah data sebelum:", len(df))

df = df.dropna(subset=["Title Cleaned"])

# Cek missing values
missing = df.info()

print("Jumlah data sesudah dropna:", len(df))

Jumlah data sebelum: 14945
<class 'pandas.DataFrame'>
Index: 14925 entries, 0 to 14944
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Title                14925 non-null  str   
 1   Ingredients          14925 non-null  str   
 2   Steps                14925 non-null  str   
 3   Loves                14925 non-null  int64 
 4   URL                  14925 non-null  str   
 5   Category             14925 non-null  str   
 6   Title Cleaned        14925 non-null  str   
 7   Total Ingredients    14925 non-null  int64 
 8   Ingredients Cleaned  14925 non-null  object
 9   Total Steps          14925 non-null  int64 
dtypes: int64(3), object(1), str(6)
memory usage: 1.3+ MB
Jumlah data sesudah dropna: 14925


In [ ]:
# Sentence case kolom Title dan Title Cleaned
df["Title"] = df["Title"].str.title()
df["Title Cleaned"] = df["Title Cleaned"].str.title()
print(df[['Title', 'Title Cleaned']].head(10).to_string())

                           Title                  Title Cleaned
0               Ayam Woku Manado               Ayam Woku Manado
1       Ayam Goreng Tulang Lunak       Ayam Goreng Tulang Lunak
2               Ayam Cabai Kawin               Ayam Cabai Kawin
3                    Ayam Geprek                    Ayam Geprek
4                    Minyak Ayam                    Minyak Ayam
5                Nasi Bakar Ayam                Nasi Bakar Ayam
6        Ayam Saus Hintalu Jaruk        Ayam Saus Hintalu Jaruk
7  Ayam Saos Teriyaki Lada Hitam  Ayam Saos Teriyaki Lada Hitam
8                     Steak Ayam                     Steak Ayam
9    Ayam Saos Asam Manis Simple    Ayam Saos Asam Manis Simple


In [ ]:
# Isi 'Title Cleaned' — update semua baris (bukan hanya NaN)
df['Title Cleaned'] = df['Title'].apply(clean_title)

print("NaN Title Cleaned:", df['Title Cleaned'].isna().sum())
print("Contoh hasil:")
print(df[['Title', 'Title Cleaned']].head(10).to_string())

NaN Title Cleaned: 0
Contoh hasil:
                           Title                  Title Cleaned
0               Ayam Woku Manado               ayam woku manado
1       Ayam Goreng Tulang Lunak       ayam goreng tulang lunak
2               Ayam Cabai Kawin               ayam cabai kawin
3                    Ayam Geprek                    ayam geprek
4                    Minyak Ayam                    minyak ayam
5                Nasi Bakar Ayam                nasi bakar ayam
6        Ayam Saus Hintalu Jaruk        ayam saus hintalu jaruk
7  Ayam Saos Teriyaki Lada Hitam  ayam saos teriyaki lada hitam
8                     Steak Ayam                     steak ayam
9    Ayam Saos Asam Manis Simple           ayam saos asam manis


In [ ]:
df.to_csv("data_cleaned.csv", index=False, encoding="utf-8-sig")
print("Tersimpan → data_cleaned.csv")
print("Shape:", df.shape)

Tersimpan → data_cleaned.csv
Shape: (14925, 10)


In [ ]:
df = pd.read_csv("data_cleaned.csv")

# Kembalikan string list → list asli
def convert_to_list(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

df["Ingredients Cleaned"] = df["Ingredients Cleaned"].apply(convert_to_list)
print("Sample Ingredients Cleaned:", df["Ingredients Cleaned"].iloc[0])

Sample Ingredients Cleaned: ['ekor ayam', 'jeruk nipis', 'garam', 'kunyit', 'bawang merah', 'bawang putih', 'cabe merah', 'cabe rawit', 'kemiri', 'sereh', 'daun salam', 'ikat daun', 'penyedap rasa', 'gelas air']


In [ ]:
def clean_phrase_level(items):
    """Normalisasi frasa & hapus noise pada level list bahan."""
    result = []

    for item in items:
        item = str(item).lower().strip()

        # skip phrase noise penuh
        if item in REMOVE_PHRASES:
            continue

        # normalisasi frasa
        for old, new in NORMALIZATION.items():
            if old in item:
                item = item.replace(old, new)

        # hapus word noise
        words = [w for w in item.split() if w not in STOPWORDS]
        item = " ".join(words).strip()

        if not item or len(item) <= 2:
            continue

        result.append(item)

    return list(dict.fromkeys(result))   # hapus duplikat, pertahankan urutan


df["Ingredients Final"] = df["Ingredients Cleaned"].apply(clean_phrase_level)
print(df["Ingredients Final"].head(3))

0    [ayam, jeruk nipis, garam, kunyit, bawang mera...
1    [ayam, serai, jeruk, bawang putih, ketumbar, l...
2    [ayam, cabai hijau, cabai merah, bawang putih,...
Name: Ingredients Final, dtype: object


In [ ]:
# Multi-kata penting → single token (agar TF-IDF tidak pecah)
PHRASE_MAP = {
    "bawang putih": "bawang_putih",
    "bawang merah": "bawang_merah",
    "cabai rawit": "cabai_rawit",
    "cabai merah": "cabai_merah",
    "cabai hijau": "cabai_hijau",
    "jeruk nipis": "jeruk_nipis",
    "saus tiram": "saus_tiram",
    "saus sambal": "saus_sambal",
    "kecap manis": "kecap_manis",
    "daun salam": "daun_salam",
    "daun jeruk": "daun_jeruk",
    "minyak goreng": "minyak_goreng",
    "tepung tapioka": "tepung_tapioka",
    "tepung bumbu": "tepung_bumbu",
    "fillet ayam": "fillet_ayam",
    "ceker ayam": "ceker_ayam",
    "daging ayam": "daging_ayam",
}

NORMALISASI_FINAL = {
    "daging_ayam": "ayam",
    "bawang_daun": "daun_bawang",
}

REMOVE_SINGLES = {"bawang", "daun"}


def apply_phrase_map(lst):
    result = []
    for item in lst:
        # terapkan phrase map
        for old, new in PHRASE_MAP.items():
            item = item.replace(old, new)
        # ganti spasi → underscore untuk multi-kata yang tersisa
        item = item.replace(' ', '_')
        # normalisasi akhir
        item = NORMALISASI_FINAL.get(item, item)
        # skip kata tunggal noise
        if item in REMOVE_SINGLES:
            continue
        result.append(item)
    return list(dict.fromkeys(result))


df["Ingredients Final"] = df["Ingredients Final"].apply(apply_phrase_map)
df["Ingredients Join"] = df["Ingredients Final"].apply(lambda x: " ".join(x))
print(df["Ingredients Join"].head(3))

0    ayam jeruk_nipis garam kunyit bawang_merah baw...
1    ayam serai jeruk bawang_putih ketumbar laos ku...
2    ayam cabai_hijau cabai_merah bawang_putih bawa...
Name: Ingredients Join, dtype: str


In [ ]:
print("Jumlah data sebelum drop duplikat:", len(df))

df = df[df["Ingredients Join"].str.strip() != ""]
df = df.drop_duplicates(subset=["Title Cleaned", "Ingredients Join"])
df = df.reset_index(drop=True)

print("Jumlah data sesudah drop duplikat:", len(df))

# Top 50 bahan paling sering muncul
all_words = " ".join(df["Ingredients Join"]).split()
freq = Counter(all_words)
print("\nTop 50 bahan:", freq.most_common(50))

Jumlah data sebelum drop duplikat: 14925
Jumlah data sesudah drop duplikat: 14915

Top 50 bahan: [('bawang_putih', 11664), ('garam', 10365), ('bawang_merah', 8581), ('minyakk', 4551), ('gula', 4528), ('bumbu', 3953), ('cabai_rawit', 3704), ('cabai_merah', 3166), ('jahe', 3045), ('merica', 2933), ('salam', 2567), ('kunyit', 2436), ('lada', 2354), ('jeruk', 2323), ('ketumbar', 2311), ('tomat', 2279), ('kemiri', 2273), ('kecap_manis', 2131), ('telur', 2026), ('lengkuas', 2017), ('bawang_bombay', 1808), ('kecap', 1702), ('saus_tiram', 1348), ('udang', 1335), ('penyedap', 1310), ('wortel', 1178), ('serai', 1140), ('kaldu', 1139), ('gula_pasir', 1101), ('papan_tempe', 1090), ('ayam', 1081), ('sereh', 1047), ('daging_sapi', 968), ('santan', 892), ('cabai', 885), ('tepung_terigu', 867), ('daging_kambing', 848), ('telur_ayam', 832), ('bh_cabai', 768), ('jeruk_nipis', 739), ('tepung', 737), ('cabai_hijau', 715), ('daging', 708), ('tahu', 669), ('kentang', 659), ('lbr', 654), ('gula_merah', 584),

In [ ]:
df_final = df[["Title Cleaned", "Ingredients Final", "Ingredients Join"]].copy()

print(df_final.head(10))
print("\nShape:", df_final.shape)

                   Title Cleaned  \
0               ayam woku manado   
1       ayam goreng tulang lunak   
2               ayam cabai kawin   
3                    ayam geprek   
4                    minyak ayam   
5                nasi bakar ayam   
6        ayam saus hintalu jaruk   
7  ayam saos teriyaki lada hitam   
8                     steak ayam   
9           ayam saos asam manis   

                                   Ingredients Final  \
0  [ayam, jeruk_nipis, garam, kunyit, bawang_mera...   
1  [ayam, serai, jeruk, bawang_putih, ketumbar, l...   
2  [ayam, cabai_hijau, cabai_merah, bawang_putih,...   
3  [ayam, gula, tepung_ayam, lalapan, kol, timun,...   
4      [ayam, bawang_putih, jahe, minyakk, ketumbar]   
5  [piring_nasi, fillet_ayam, kotak, bersih, pisa...   
6  [ayam, hintalu_jaruk, cabai_merah, cabai_hijau...   
7  [ayam, bawang_bombay, bawang_putih, cabai_mera...   
8  [dada_ayam, jeruk, garam, adonan_basah, terigu...   
9  [ayam, tepung, bawang_putih, bawang_mera

In [ ]:
df.to_csv("data_final.csv", index=False, encoding="utf-8-sig")
print("Tersimpan → data_final.csv")
print("Kolom:", df.columns.tolist())
print("Shape:", df.shape)

Tersimpan → data_final.csv
Kolom: ['Title', 'Ingredients', 'Steps', 'Loves', 'URL', 'Category', 'Title Cleaned', 'Total Ingredients', 'Ingredients Cleaned', 'Total Steps', 'Ingredients Final', 'Ingredients Join']
Shape: (14915, 12)


In [ ]:
# Cek missing values
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 14915 entries, 0 to 14914
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Title Cleaned      14915 non-null  str   
 1   Ingredients Final  14915 non-null  object
 2   Ingredients Join   14915 non-null  str   
dtypes: object(1), str(2)
memory usage: 349.7+ KB
